In [1]:
!pip install joblib

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

ModuleNotFoundError: No module named 'matplotlib'

In [3]:
df = pd.read_excel("furniture_sales_dataset_1500_rows_clean.xlsx")

df.columns = df.columns.str.strip()

print("Original Dataset Shape:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'furniture_sales_dataset_1500_rows_clean.xlsx'

In [ ]:
month_map = {
    "Jan":1,"January":1,
    "Feb":2,"February":2,
    "Mar":3,"March":3,
    "Apr":4,"April":4,
    "May":5,"May":5,
    "Jun":6,"June":6,
    "Jul":7,"July":7,
    "Aug":8,"August":8,
    "Sep":9,"September":9,
    "Oct":10,"October":10,
    "Nov":11,"November":11,
    "Dec":12,"December":12
}

df["Month"] = df["Month"].replace(month_map)

/tmp/ipykernel_23102/3198340531.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Month"] = df["Month"].replace(month_map)


In [ ]:
numeric_cols = ["Year","Month","Units_Sold","Price","Total_Sales"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Year"].fillna(df["Year"].median(), inplace=True)
df["Month"].fillna(df["Month"].mode()[0], inplace=True)
df["Units_Sold"].fillna(df["Units_Sold"].median(), inplace=True)
df["Price"].fillna(df["Price"].median(), inplace=True)
df["Total_Sales"].fillna(df["Total_Sales"].median(), inplace=True)

df["Product_Name"].fillna("Unknown_Product", inplace=True)
df["Category"].fillna("Unknown_Category", inplace=True)
df["Season"].fillna("Unknown_Season", inplace=True)

df["Year"] = df["Year"].astype(int)
df["Month"] = df["Month"].astype(int)

print("Dataset After Cleaning:", df.shape)

Dataset After Cleaning: (1500, 10)


/tmp/ipykernel_23102/307671304.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Year"].fillna(df["Year"].median(), inplace=True)
/tmp/ipykernel_23102/307671304.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [ ]:
df["Month_sin"] = np.sin(2*np.pi*df["Month"]/12)
df["Month_cos"] = np.cos(2*np.pi*df["Month"]/12)

df["Year_trend"] = df["Year"] - df["Year"].min()

df["Price_log"] = np.log1p(df["Price"])

In [ ]:
le_product = LabelEncoder()
le_category = LabelEncoder()
le_season = LabelEncoder()

df["Product_enc"] = le_product.fit_transform(df["Product_Name"])
df["Category_enc"] = le_category.fit_transform(df["Category"])
df["Season_enc"] = le_season.fit_transform(df["Season"])

In [ ]:
features = [
    "Year_trend",
    "Month",
    "Month_sin",
    "Month_cos",
    "Price_log",
    "Product_enc",
    "Category_enc",
    "Season_enc"
]

target = "Total_Sales"

X = df[features]
y = df[target]

print("Training Data Shape:", X.shape)

Training Data Shape: (1500, 8)


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Model Accuracy (R2 Score):", r2_score(y_test,pred))

Model Accuracy (R2 Score): 0.8094621809039384


In [ ]:
joblib.dump(model,"sales_model.pkl")
joblib.dump(le_product,"product_encoder.pkl")
joblib.dump(le_category,"category_encoder.pkl")
joblib.dump(le_season,"season_encoder.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
model = joblib.load("sales_model.pkl")
le_product = joblib.load("product_encoder.pkl")
le_category = joblib.load("category_encoder.pkl")
le_season = joblib.load("season_encoder.pkl")

print("Model Loaded Successfully")

Model Loaded Successfully


In [ ]:
products = df["Product_Name"].unique()

In [ ]:
# ==========================================================
# YEAR FORECAST
# ==========================================================

def forecast_year(year):

    results=[]

    for p in products:

        pdata=df[df["Product_Name"]==p]

        price=float(pdata["Price"].mean())
        category=pdata["Category"].iloc[0]
        season=pdata["Season"].iloc[0]

        total_units=0
        total_sales=0
        total_profit=0

        for month in range(1,13):

            year_trend=year-df["Year"].min()

            row=pd.DataFrame({

                "Year_trend":[year_trend],
                "Month":[month],
                "Month_sin":[np.sin(2*np.pi*month/12)],
                "Month_cos":[np.cos(2*np.pi*month/12)],
                "Price_log":[np.log1p(price)],
                "Product_enc":[le_product.transform([p])[0]],
                "Category_enc":[le_category.transform([category])[0]],
                "Season_enc":[le_season.transform([season])[0]]

            })

            pred_sales=float(model.predict(row)[0])

            units=pred_sales/price
            profit=pred_sales*0.25

            total_units+=units
            total_sales+=pred_sales
            total_profit+=profit

        results.append([
            p,
            round(total_units),
            round(total_sales),
            round(total_profit)
        ])

    result_df=pd.DataFrame(results,columns=[
        "Product",
        "Predicted_Units",
        "Predicted_Sales",
        "Profit"
    ])

    print("\nFull Year Product Forecast\n")
    print(result_df)

    return result_df


# ==========================================================
# PRODUCT PREDICTION
# ==========================================================

def product_forecast(product,year):

    pdata=df[df["Product_Name"]==product]

    if len(pdata)==0:

        print("Product not found")
        print("Available Products:",products)
        return

    price=float(pdata["Price"].mean())
    category=pdata["Category"].iloc[0]
    season=pdata["Season"].iloc[0]

    total_units=0
    total_sales=0
    total_profit=0

    for month in range(1,13):

        year_trend=year-df["Year"].min()

        row=pd.DataFrame({

            "Year_trend":[year_trend],
            "Month":[month],
            "Month_sin":[np.sin(2*np.pi*month/12)],
            "Month_cos":[np.cos(2*np.pi*month/12)],
            "Price_log":[np.log1p(price)],
            "Product_enc":[le_product.transform([product])[0]],
            "Category_enc":[le_category.transform([category])[0]],
            "Season_enc":[le_season.transform([season])[0]]

        })

        pred_sales=float(model.predict(row)[0])

        units=pred_sales/price
        profit=pred_sales*0.25

        total_units+=units
        total_sales+=pred_sales
        total_profit+=profit

    print("\nProduct Prediction\n")

    print("Product:",product)
    print("Year:",year)
    print("Predicted Units Sold:",round(total_units))
    print("Predicted Sales:",round(total_sales))
    print("Predicted Profit:",round(total_profit))


# ==========================================================
# COMBO RECOMMENDATION
# ==========================================================

# ==========================================================
# COMBO RECOMMENDATION BASED ON PREDICTED SALES
# ==========================================================

def combo_offer(year,month):

    predictions=[]

    for p in products:

        pdata=df[df["Product_Name"]==p]

        price=float(pdata["Price"].mean())
        category=pdata["Category"].iloc[0]
        season=pdata["Season"].iloc[0]

        year_trend=year-df["Year"].min()

        row=pd.DataFrame({

            "Year_trend":[year_trend],
            "Month":[month],
            "Month_sin":[np.sin(2*np.pi*month/12)],
            "Month_cos":[np.cos(2*np.pi*month/12)],
            "Price_log":[np.log1p(price)],
            "Product_enc":[le_product.transform([p])[0]],
            "Category_enc":[le_category.transform([category])[0]],
            "Season_enc":[le_season.transform([season])[0]]

        })

        pred_sales=float(model.predict(row)[0])

        predictions.append([p,pred_sales])

    pred_df=pd.DataFrame(predictions,columns=["Product","Predicted_Sales"])

    # highest selling products
    high=pred_df.sort_values("Predicted_Sales",ascending=False).head(5)

    # lowest selling products
    low=pred_df.sort_values("Predicted_Sales").head(5)

    combos=[]

    for h,l in zip(high["Product"],low["Product"]):

        offer=np.random.randint(15,30)

        combos.append([h,l,str(offer)+"%"])

    combo_df=pd.DataFrame(combos,columns=[
        "Main_Product","Combo_Product","Offer"
    ])

    print("\nCombo Offers Based on Predicted Sales\n")
    print("Year:",year,"Month:",month)
    print(combo_df)

    print("\nRecommended Combo Offers\n")
    print(combo_df)

In [ ]:
# ==========================================================
# MENU SYSTEM
# ==========================================================

while True:

    print("\nAI Furniture Sales Prediction System")

    print("1 - Forecast Full Year")
    print("2 - Product Prediction")
    print("3 - Combo Offer Recommendation")
    print("4 - Exit")

    choice=input("Enter choice: ")

    if choice=="1":

        year=int(input("Enter Year: "))
        forecast_year(year)

    elif choice=="2":

        product=input("Enter Product Name: ")
        year=int(input("Enter Year: "))
        product_forecast(product,year)

    elif choice=="3":

        year=int(input("Enter Year: "))
        month=int(input("Enter Month (1-12): "))
        combo_offer(year,month)

    elif choice=="4":

        print("System Closed")
        break

    else:

        print("Invalid Choice")


AI Furniture Sales Prediction System
1 - Forecast Full Year
2 - Product Prediction
3 - Combo Offer Recommendation
4 - Exit
Enter choice: 3
Enter Year: 2027
Enter Month (1-12): 5

Combo Offers Based on Predicted Sales

Year: 2027 Month: 5
      Main_Product  Combo_Product Offer
0               AC  Plastic Chair   15%
1  Washing Machine       Iron Box   20%
2             Sofa         Heater   19%
3     Dining Table          Chair   17%
4           Fridge     Cloth Rack   15%

Recommended Combo Offers

      Main_Product  Combo_Product Offer
0               AC  Plastic Chair   15%
1  Washing Machine       Iron Box   20%
2             Sofa         Heater   19%
3     Dining Table          Chair   17%
4           Fridge     Cloth Rack   15%

AI Furniture Sales Prediction System
1 - Forecast Full Year
2 - Product Prediction
3 - Combo Offer Recommendation
4 - Exit
Enter choice: 4
System Closed
